# CUHK-X Large Model Track — Colab 실행 노트북

워크플로: **[로컬 VS Code] 코드 수정 → git push → [이 노트북] Runtime > Run all**

- 모든 셀은 **재실행 가능(idempotent)** — VM이 리셋되면 그냥 위에서부터 다시 실행
- 결과물(submission/val_pred)은 Drive에 저장 → 세션이 끊겨도 resume 됨
- 런타임 유형: **L4 이상**(7B), T4면 아래 MODEL을 3B로

In [ ]:
# ===== 0. 설정 (본인 저장소 주소로 수정) =====
REPO_URL  = "https://github.com/<YOUR_GITHUB_ID>/cuhk_euron.git"
DRIVE_OUT = "/content/drive/MyDrive/cuhk"          # 결과 저장 위치 (Drive)
MODEL     = "Qwen/Qwen2.5-VL-7B-Instruct"          # T4(16GB)면 3B로 변경
MODALITY  = "IR"                                    # IR이 가장 선명 (Depth_Color도 실험 가치)

!nvidia-smi -L

In [ ]:
# ===== 1. Drive 마운트 (결과 저장/resume 용) =====
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs(DRIVE_OUT, exist_ok=True)

In [ ]:
# ===== 2. 코드 받기 (없으면 clone, 있으면 pull) =====
import os
if not os.path.exists('/content/cuhk_euron'):
    !git clone {REPO_URL} /content/cuhk_euron
%cd /content/cuhk_euron
!git pull

In [ ]:
# ===== 3. 의존성 설치 =====
!pip install -q -r requirements.txt huggingface_hub

In [ ]:
# ===== 4. 데이터 다운로드 (HF → VM 로컬 디스크) + 압축 해제 =====
# 실제 레이아웃: Training/{training_qa.csv, modality_list.csv, data/HARn.zip, data/HAU.zip}
#               Testing/{test_qa.csv, data/large_model_track_test.zip}
from huggingface_hub import snapshot_download
snapshot_download('Kevin-Pal/CUHK-X_Large_Model_Track', repo_type='dataset',
                  local_dir='data')

# zip은 어디에 있든 전부 data/ 바로 아래로 해제 (unzip -n: 이미 풀린 건 skip)
!find data -name "*.zip" -exec unzip -qn {} -d data \;

# csv를 data/ 루트로 복사 → 스크립트 기본 인자와 일치
import glob, shutil, os
for name in ['training_qa.csv', 'test_qa.csv', 'modality_list.csv']:
    hits = [h for h in glob.glob(f'data/**/{name}', recursive=True)
            if os.path.dirname(h) != 'data']
    if hits and not os.path.exists(f'data/{name}'):
        shutil.copy(hits[0], f'data/{name}')

!ls data

In [ ]:
# ===== 5. 전처리 (선택 — 반복 실험 시 추론 속도 향상) =====
!python src/preprocess_harn.py --harn-root data/HARn --out data/cache/HARn \
    --index data/cache/harn_index.csv --frames 16 --workers 4 --modality {MODALITY}
!python src/preprocess_harn.py --harn-root data/HAU --out data/cache/HAU \
    --index data/cache/hau_index.csv --frames 16 --workers 4 --modality {MODALITY}

In [ ]:
# ===== 6. 로컬 검증 (hold-out user 9, 24) =====
# 먼저 --limit 100으로 빠르게 확인하고, 괜찮으면 limit 제거
!python src/run_baseline.py --qa data/training_qa.csv \
    --out {DRIVE_OUT}/val_pred.csv --val-users 9,24 --limit 100 \
    --media-root data/cache,data --model {MODEL} --modality {MODALITY}

!python src/evaluate.py --pred {DRIVE_OUT}/val_pred.csv --gold data/training_qa.csv

In [ ]:
# ===== 7. 테스트 추론 → Drive에 submission 저장 =====
# 세션이 끊겨도 같은 명령을 다시 실행하면 이어서 돌아감 (resume)
!python src/run_baseline.py --qa data/test_qa.csv \
    --out {DRIVE_OUT}/submission.csv \
    --media-root data/cache,data --model {MODEL} --modality {MODALITY}

import pandas as pd
sub = pd.read_csv(f'{DRIVE_OUT}/submission.csv')
print(len(sub), 'rows')
sub.head()

완료 후 Drive의 `cuhk/submission.csv`를 Kaggle에 제출.

**팁**
- 코드 수정은 로컬 VS Code에서 → push → 여기서는 **셀 2(git pull)부터** 다시 실행
- HF 다운로드가 느린 날은: zip들을 Drive에 한 번 복사해 두고, 셀 4를 `Drive → /content 복사 + unzip`으로 교체
- modality 실험: 셀 0의 `MODALITY`를 `Depth_Color`로 바꿔 재실행 후 검증 점수 비교
- 검증 정확도는 category별로 보고 병목(multi/sequence)부터 개선 (PIPELINE.md 7절 로드맵)